# 04 · 评估、导出与部署

从训练产物到可用服务。**配置与规划部分不需要 GPU**，只有真正加载模型时才需要。

## 1. 评估

`ModelEvaluator(model, tokenizer, device)` 需要真实的模型与分词器对象，
因此这一节只展示配置部分，真实调用方式见代码注释。

困惑度（perplexity）衡量"模型对数据的惊讶程度"，**越低越好**；
但跨词表不可比 —— 只用来对比**同一基座**的不同微调版本。

In [ ]:
import sys

sys.path.insert(0, "..")

from llm_finetuning import Config, EvaluationConfig, ModelEvaluator

config = Config(model_name_or_path="Qwen/Qwen2.5-0.5B-Instruct")
config.evaluation = EvaluationConfig(metrics=["perplexity"], max_new_tokens=64)
print("评估配置:", config.evaluation.to_dict())

print("\n真实调用方式（需要已加载的模型与分词器）:")
print("    evaluator = ModelEvaluator(model, tokenizer, device='auto')")
print("    result = evaluator.evaluate(test_dataset)")
print(f"\nModelEvaluator 签名: {ModelEvaluator.__init__.__doc__.strip().splitlines()[0]}")

## 2. 导出

三种目标格式各有归属：

| 格式 | 用途 | 依赖 |
|---|---|---|
| **safetensors** | 继续训练 / HF 生态 | 无额外依赖 |
| **ONNX** | 生产推理、跨框架 | `onnx` + `optimum` |
| **GGUF** | llama.cpp / Ollama 本地部署 | llama.cpp 转换脚本 |

> GGUF 无法用纯 Python 生成。项目的做法是：先落盘 HF 权重，再返回一条
> **可直接执行的转换命令**，不做假装实现。

In [ ]:
from llm_finetuning.export import ExportConfig, ExportFormat, ModelExporter

exp_cfg = ExportConfig(
    output_dir="./outputs/export",
    export_format=ExportFormat.SAFETENSORS,
    quantization="fp16",
)
exp_cfg.validate()  # 非法取值在导出前就抛错，避免跑到一半才失败
print("导出配置:", exp_cfg)

exporter = ModelExporter(exp_cfg)
print("\n真实调用方式（需要模型对象）:")
print("    result = exporter.export(model, tokenizer)")

### 体积估算

`ModelExporter.estimate_size_mb` 是静态方法，**不需要模型**，
可在选型阶段直接估算目标设备放不放得下。

In [ ]:
for params in (1_500_000_000, 7_000_000_000, 14_000_000_000):
    sizes = {
        level: f"{ModelExporter.estimate_size_mb(params, level):.0f} MB"
        for level in ("fp16", "int8", "int4")
    }
    print(f"{params / 1e9:>3.0f}B  {sizes}")

## 3. 部署

`deploy/server.py` 提供 FastAPI 服务，支持两种后端：

- **transformers**：依赖少、易调试，吞吐一般
- **vLLM**：PagedAttention + 连续批处理，吞吐高，生产首选

下面只构建配置与后端对象（不加载模型、不发请求）。

In [ ]:
from llm_finetuning.deploy import DeployConfig, build_backend

deploy_cfg = DeployConfig(
    model_path="./outputs/export",
    backend="transformers",
    host="0.0.0.0",
    port=8000,
    max_new_tokens=256,
)
print("部署配置:", deploy_cfg)

try:
    backend = build_backend(deploy_cfg)
    print("后端已构建:", type(backend).__name__)
except Exception as exc:
    print(f"构建后端需要 torch/transformers: {type(exc).__name__}: {exc}")

### 启动服务

两种方式等价：

```python
from llm_finetuning.deploy import DeployConfig, run_server

run_server(DeployConfig(model_path="./outputs/export", backend="transformers", port=8000))
```

```bash
python -m llm_finetuning.deploy --model-path ./outputs/export --backend transformers --port 8000
python -m llm_finetuning.deploy --model-path ./outputs/export --backend vllm --port 8000
```

接口：

| 方法 | 路径 | 说明 |
|---|---|---|
| GET | `/health` | 健康检查 |
| GET | `/v1/info` | 模型与后端信息 |
| POST | `/v1/generate` | 文本生成，参数可请求级覆盖 |

```bash
curl -X POST http://localhost:8000/v1/generate \
     -H 'Content-Type: application/json' \
     -d '{"prompt": "用一句话解释 LoRA", "max_new_tokens": 128}'
```

> 服务依赖 `fastapi` + `uvicorn`，两个后端还需要 `torch/transformers` 或 `vllm`。